# ARISTA spatial/PCA velocity and direction correlation — CytoBridge API

**Objective.** Recompute the t1 spatial and PCA velocity fields plus the cosine similarity between full and interaction velocity inside the reaEGC-defined ROI using public package APIs.


## Plan

1. Load a published or current ARISTA checkpoint through the package loader.
2. Compute the full velocity decomposition at t1.
3. Project full and interaction fields with the shared scVelo spatial embedding API.
4. Export spatial/PCA streamlines, select the reaEGC bounding-box ROI, and save cosine values plus a manifest.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'downstream_helpers').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from downstream_helpers.arista_api import (
    AristaSpatiotemporalConfig,
    assert_package_only_runtime,
    run_arista_direction_correlation_api,
)
from downstream_helpers.runner import display_svg_outputs

DEVICE = os.environ.get('CYTOBRIDGE_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
SMOKE = os.environ.get('CYTOBRIDGE_SMOKE', '0') == '1'
MODEL_FORMAT = os.environ.get('ARISTA_MODEL_FORMAT', 'legacy')
ALIGNED_H5AD = os.environ.get('ARISTA_ALIGNED_H5AD') or None
MODEL_DIR = os.environ.get('ARISTA_MODEL_DIR') or None
DEVICE, SMOKE, MODEL_FORMAT


## Configuration

The t1 and reaEGC choices reproduce the ARISTA-specific panels. The velocity decomposition, spatial/PCA streamlines, scVelo projection, cosine calculation, ROI selection, and plotting are parameterized package functions.


In [ ]:
if MODEL_FORMAT == 'current' and (ALIGNED_H5AD is None or MODEL_DIR is None):
    raise ValueError('Current mode requires ARISTA_ALIGNED_H5AD and ARISTA_MODEL_DIR.')

config = AristaSpatiotemporalConfig(
    output_name='arista_velocity_direction_correlation_api' + ('_smoke' if SMOKE else ''),
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    model_format=MODEL_FORMAT,
    random_seed=42,
    device=DEVICE,
    run_communication=False,
    run_3d=False,
)
config


In [ ]:
result = run_arista_direction_correlation_api(
    config,
    target_timepoint=1.0,
    focus_label_keyword='reaEGC',
    pad_ratio=0.15,
    n_neighbors=30,
    max_cells=256 if SMOKE else None,
)
assert_package_only_runtime()
result


## Results

The first two outputs are the full spatial and PCA velocity fields. In the ROI panel, cosine values near +1 indicate aligned full and interaction directions; values near −1 indicate opposition. The CSV preserves every plotted ROI cell for quantitative comparison.


In [ ]:
display_svg_outputs([result.spatial_velocity_figure, result.pca_velocity_figure, result.figure_path])
print(result.roi_csv.read_text(encoding='utf-8')[:3000])
print(result.manifest_path.read_text(encoding='utf-8'))


## Next checks

- Compare ROI cosine distributions across the three model/threshold runs.
- Confirm that the selected ROI contains reaEGC cells and remains stable under the configured padding.
